In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps peft accelerate bitsandbytes xformers
!pip install -q sentence-transformers faiss-cpu openpyxl pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.7 MB/s et

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import pandas as pd
import torch

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN is missing. Add it in Colab Secrets or environment variables.')

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, go to Runtime > Change runtime type > T4 GPU.')

print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


In [ ]:
PROMPT_CSV_PATH = '/content/drive/MyDrive/dataset_clean.csv'
ANSWER_XLSX_PATH = '/content/drive/MyDrive/Austrian Tax Law LLM Prompt Assignment.xlsx'
LEGAL_DOCS_FOLDER = '/content/drive/MyDrive/legal_docs'

RAG_SUBMISSION_PATH = '/content/drive/MyDrive/rag_submission.csv'
RAG_PARTIAL_PATH = '/content/drive/MyDrive/rag_submission_partial.csv'
RAG_COMPARISON_PATH = '/content/drive/MyDrive/rag_comparison.csv'

BASELINE_RESULTS_PATH = '/content/drive/MyDrive/baseline_results.csv'
FINAL_FT_RESULTS_PATH = '/content/drive/MyDrive/fine_tuned_results.csv'

df = pd.read_csv(PROMPT_CSV_PATH)
df = df[['id', 'prompt']].dropna(subset=['prompt']).reset_index(drop=True)

answers_df = pd.read_excel(ANSWER_XLSX_PATH, sheet_name='Dataset')
answers_df = answers_df[['id', 'correct_answer', 'sources']].dropna(subset=['id'])
answers_df = answers_df.drop_duplicates(subset=['id']).reset_index(drop=True)

evaluation_df = df.merge(answers_df, on='id', how='left').reset_index(drop=True)

print(f'Loaded {len(evaluation_df)} questions.')
print(evaluation_df.head(3).to_string())


Loaded 643 questions.
             id                                                                                                                   prompt                                                                                                                                                                                                                                                                                           correct_answer                                                                  sources
0  CORP-TAX-001                                                  Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?                                                                                                                                                                                       Das Einkommen, das der unbeschränkt Steuerpflichtige innerhalb eines Kalenderjahres bezogen hat (§ 7 Abs. 1 KStG).                                      

In [ ]:
!pip install -q pypdf

from pypdf import PdfReader
import os

CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

if not os.path.isdir(LEGAL_DOCS_FOLDER):
    raise FileNotFoundError(f"Legal docs folder not found: {LEGAL_DOCS_FOLDER}")

doc_files = sorted([f for f in os.listdir(LEGAL_DOCS_FOLDER) if f.lower().endswith(".pdf")])

if not doc_files:
    raise FileNotFoundError(f"No .pdf files found in {LEGAL_DOCS_FOLDER}")

print(f"Found {len(doc_files)} legal document(s):")
for f in doc_files[:10]:
    print(" -", f)

all_chunks = []
chunk_sources = []

for filename in doc_files:
    filepath = os.path.join(LEGAL_DOCS_FOLDER, filename)

    reader = PdfReader(filepath)
    full_text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"

    words = full_text.split()
    pos = 0

    while pos < len(words):
        chunk_text = " ".join(words[pos : pos + CHUNK_SIZE])
        all_chunks.append(chunk_text)
        chunk_sources.append(filename)
        pos += CHUNK_SIZE - CHUNK_OVERLAP

print("Total chunks created:", len(all_chunks))
print("Sample chunk:")
print(all_chunks[0][:300])


Found 22 legal document(s):
 - 1.pdf
 - 10.pdf
 - 11.pdf
 - 12.pdf
 - 13.pdf
 - 14.pdf
 - 15.pdf
 - 16.pdf
 - 17.pdf
 - 18.pdf


Total chunks created: 476
Sample chunk:
Bundesrecht konsolidiert www.ris.bka.gv.at Seite 1 von 5 Kurztitel Einkommensteuergesetz 1988 Kundmachungsorgan BGBl. Nr. 400/1988 zuletzt geändert durch BGBl. I Nr. 200/2023 Typ BG §/Artikel/Anlage § 27 Inkrafttretensdatum 01.01.2024 Außerkrafttretensdatum 23.12.2025 Abkürzung EStG 1988 Index 32/02


In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

print('Loading embedding model...')
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print(f'Embedding {len(all_chunks)} chunks...')
t0 = time.time()
chunk_vecs = embed_model.encode(
    all_chunks,
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64,
)
print(f'Embedding done in {time.time() - t0:.1f}s.')

faiss.normalize_L2(chunk_vecs)
dim = chunk_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_vecs.astype('float32'))

print(f'FAISS index built: {index.ntotal} vectors, dimension {dim}.')


Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 476 chunks...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding done in 2.8s.
FAISS index built: 476 vectors, dimension 384.


In [ ]:
TOP_K = 3
questions = evaluation_df['prompt'].tolist()

print(f'Embedding {len(questions)} questions...')
q_vecs = embed_model.encode(questions, convert_to_numpy=True, batch_size=64, show_progress_bar=True)
faiss.normalize_L2(q_vecs)

print(f'Searching index for top-{TOP_K} chunks per question...')
distances, indices_rag = index.search(q_vecs.astype('float32'), k=TOP_K)

print('Retrieval complete.')
print('Example retrieval:')
print(questions[0])
for rank in range(TOP_K):
    ci = indices_rag[0][rank]
    print(f'Chunk {rank+1} from {chunk_sources[ci]}:')
    print(all_chunks[ci][:150])
    print()


Embedding 643 questions...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Searching index for top-3 chunks per question...
Retrieval complete.
Example retrieval:
Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?
Chunk 1 from 4.pdf:
buchmäßig nachgewiesen sein Steuersätze (1)Die Steuer beträgt für jeden steuerpflichtigen Umsatz 20% der Bemessungsgrundlage (§§ 4 und 5). (2)Die Steu

Chunk 2 from 11.pdf:
4a des Einkommensteuergesetzes 1988 oder nach § 8 Abs. 4 Z 1 abzugsfähig sind. 6.Die Steuern vom Einkommen und sonstige Personensteuern und die aus An

Chunk 3 from 13.pdf:
Urkunde dokumentiert sein. (3)Ein gemäß § 201 BAO festgesetzter Steuerbetrag hat den im Abs. 1 genannten Fälligkeitstag. Die selbstzuberechnende Steue



In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)

from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct-bnb-4bit',
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    token=HF_TOKEN,
)

FastLanguageModel.for_inference(model)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token

print('Generation model loaded and ready.')


/tmp/ipykernel_8972/1147182645.py:4: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


Generation model loaded and ready.


In [ ]:
RAG_SYSTEM_MSG = (
    'Du bist ein Experte fuer oesterreichisches Steuerrecht. '
    'Dir werden relevante Auszuege aus oesterreichischen Steuergesetzen und eine Frage gegeben. '
    'Beantworte die Frage kurz und praezise auf Deutsch, gestuetzt auf die gegebenen Gesetzestexte.'
)

def build_chat_text(system_msg, user_msg):
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{system_msg}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'{user_msg}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )

rag_prompts = []
for i, q in enumerate(questions):
    context_parts = [all_chunks[indices_rag[i][rank]] for rank in range(TOP_K)]
    context = '\n\n---\n\n'.join(context_parts)
    user_msg = f'Relevante Gesetzestexte:\n\n{context}\n\nFrage: {q}'
    rag_prompts.append(build_chat_text(RAG_SYSTEM_MSG, user_msg))

print(f'Built {len(rag_prompts)} RAG prompts.')
print('Sample prompt length:', len(rag_prompts[0]))


Built 643 RAG prompts.
Sample prompt length: 7529


In [ ]:
import torch

BATCH_SIZE = 4
MAX_NEW_TOKENS = 120

rag_answers = []
t0 = time.time()

print(f'Running RAG inference on {len(rag_prompts)} questions (batch size {BATCH_SIZE})...')

for i in range(0, len(rag_prompts), BATCH_SIZE):
    batch = rag_prompts[i : i + BATCH_SIZE]

    inputs = tokenizer(
        batch,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    input_len = inputs['input_ids'].shape[1]
    for out in outputs:
        answer = tokenizer.decode(out[input_len:], skip_special_tokens=True).strip()
        rag_answers.append(answer)

    done = min(i + BATCH_SIZE, len(rag_prompts))
    if done % 50 == 0 or done == len(rag_prompts):
        partial_df = pd.DataFrame({
            'id': evaluation_df['id'].iloc[:done].tolist(),
            'answer': rag_answers,
        })
        partial_df.to_csv(RAG_PARTIAL_PATH, index=False)
        elapsed = time.time() - t0
        print(f'  {done}/{len(rag_prompts)} | {elapsed:.0f}s')

print(f'RAG inference complete in {time.time() - t0:.0f}s.')


Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running RAG inference on 643 questions (batch size 4)...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

  100/643 | 172s


Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  500/643 | 776s


Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  600/643 | 937s


Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  643/643 | 1006s
RAG inference complete in 1006s.


In [ ]:
rag_submission = pd.DataFrame({
    'id': evaluation_df['id'].tolist(),
    'answer': rag_answers,
})

rag_submission.to_csv(RAG_SUBMISSION_PATH, index=False)

print(f'Saved: {RAG_SUBMISSION_PATH}')
print(f'Total rows: {len(rag_submission)}')
rag_submission.head()


Saved: /content/drive/MyDrive/rag_submission.csv
Total rows: 643


,id,answer
0,CORP-TAX-001,"an Personenvereinigungen, die nicht mit der Üb..."
1,CORP-TAX-002,äge abzugsfrei.
2,CORP-TAX-003,kann die Anforderungen der Körperschaftsteuere...
3,CORP-TAX-004,"Leistungen, die regelmäßig mit dem Betrieb ein..."
4,CORP-TAX-005,", die an Personen, die für die Überwachung der..."


In [ ]:
comparison_df = evaluation_df.copy()

if os.path.exists(BASELINE_RESULTS_PATH):
    baseline_df = pd.read_csv(BASELINE_RESULTS_PATH)
    if 'baseline_answer' not in baseline_df.columns and 'answer' in baseline_df.columns:
        baseline_df = baseline_df.rename(columns={'answer': 'baseline_answer'})
    baseline_df = baseline_df[['id', 'baseline_answer']].drop_duplicates(subset=['id'])
    comparison_df = comparison_df.merge(baseline_df, on='id', how='left')

if os.path.exists(FINAL_FT_RESULTS_PATH):
    ft_df = pd.read_csv(FINAL_FT_RESULTS_PATH)
    if 'tuned_answer' not in ft_df.columns and 'answer' in ft_df.columns:
        ft_df = ft_df.rename(columns={'answer': 'tuned_answer'})
    ft_df = ft_df[['id', 'tuned_answer']].drop_duplicates(subset=['id'])
    comparison_df = comparison_df.merge(ft_df, on='id', how='left')

comparison_df = comparison_df.merge(
    rag_submission.rename(columns={'answer': 'rag_answer'}),
    on='id',
    how='left'
)

comparison_df.to_csv(RAG_COMPARISON_PATH, index=False)
print(f'Saved: {RAG_COMPARISON_PATH}')
comparison_df.head()


Saved: /content/drive/MyDrive/rag_comparison.csv


,id,prompt,correct_answer,sources,baseline_answer,tuned_answer,rag_answer
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...,"Das Einkommen, das der unbeschränkt Steuerpfli...",§ 7 Abs. 1 KStG 1988,Die steuerliche Bemessungsgrundlage für die Kö...,Die Bemessungsgrundlage für Körperschaftsteuer...,"an Personenvereinigungen, die nicht mit der Üb..."
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ...",Die nicht verrechneten Zinsen stellen eine ver...,§ 7 Abs. 2/3 KStG 1988; § 8 Abs. 2 KStG 1988; ...,Das zinslose Darlehen an den Gesellschafter fü...,"Die Darlehen sind steuerfrei, da sie nicht als...",äge abzugsfrei.
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl...","Steuerpflichtige, die aufgrund ihrer Rechtsfor...",§ 7 Abs. 3 KStG 1988,"Körperschaften, die verpflichtet sind, sämtlic...","Körperschaften, die Gewerbebetrieb betreiben, ...",kann die Anforderungen der Körperschaftsteuere...
3,CORP-TAX-004,Wie wird eine Dividende einer österreichischen...,(a) Tochter: Die Ausschüttung ist Einkommensve...,§ 8 Abs. 2 KStG 1988; § 10 Abs. 1 Z 1 KStG 1988,(a) Bei der Tochtergesellschaft wird die Divid...,a) Bei der Tochtergesellschaft: Die Dividende ...,"Leistungen, die regelmäßig mit dem Betrieb ein..."
4,CORP-TAX-005,Was unterscheidet die steuerliche Behandlung e...,Im Ergebnis unterscheiden sie sich nicht. Beid...,§ 8 Abs. 2 KStG 1988,Eine offene Ausschüttung ist eine direkte Zahl...,"Die Unterscheidung liegt in der Art und Weise,...",", die an Personen, die für die Überwachung der..."
